# T1-U7BC-M1 — matched-example audit

This notebook reproduces an inference-only comparison of U7B and U7C on both original evaluation suites. Upload `T1_U7BC_Matched_Audit.zip` when prompted. The ZIP contains compact references and all twelve frozen checkpoints. No model training, probes, or confirmation seeds are run. CPU is sufficient.

The original suites are identical for seeds 503/607/709. They are labeled as duplicate rows, not independent repetitions. All six exposed seeds retain their original gates.

The package verifies its original input manifest before inference and checks all twelve native endpoints. Reproduction writes to a new `reproduced_results` directory, preserving distributed results.

In [ ]:
from pathlib import Path
from google.colab import files
import zipfile
import subprocess
import sys

uploaded = files.upload()
archive = Path('T1_U7BC_Matched_Audit.zip')
if not archive.is_file():
    raise FileNotFoundError('Upload T1_U7BC_Matched_Audit.zip')
with zipfile.ZipFile(archive) as z:
    if any(Path(n).is_absolute() or '..' in Path(n).parts for n in z.namelist()):
        raise ValueError('Unsafe archive path')
    z.extractall('audit_workspace')
root = Path('audit_workspace/T1_U7BC_Matched_Audit').resolve()
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(root / 'requirements.txt')], check=True)

In [ ]:
import hashlib
import json
import subprocess
import sys

manifest = json.loads((root / 'PACKAGE_SHA256.json').read_text())
for name, expected in manifest.items():
    actual = hashlib.sha256((root / name).read_bytes()).hexdigest()
    if actual != expected:
        raise ValueError(f'Package integrity check failed: {name}')
out = root / 'reproduced_results'
subprocess.run([sys.executable, '-u', str(root / 'run_matched_audit.py'), '--out', str(out)], check=True)

In [ ]:
import pandas as pd
result = json.loads((out / 'audit_result.json').read_text())
print('Status:', result['status'])
print('Native endpoints reproduced:', len(result['native_endpoint_checks']))
print('Training updates:', result['training_updates'])
print('Probe fits:', result['probe_fits'])
print('T1-U8 advancement:', result['advance_to_t1_u8'])
display(pd.DataFrame(result['suite_decisions']))
display(pd.read_csv(out / 'paired_effects.csv'))